# Notebook 03 v0.2: Session Strategy and Prompt Bundle

**Date**: 2026-04-14  
**Status**: Now uses shared importable modules from `src/intake/`

This notebook is now a **thin demo wrapper**. The core implementation has been moved to:
- `src/intake/strategy.py` — SessionStrategy, PromptBundle, and strategy builders
- `src/intake/safety.py` — Structural traveler-safe sanitization

**Contract**: see `notebooks/NB03_V02_SPEC.md`

## Imports from Shared Modules

All NB03 functionality is now importable from the `intake` package.

In [ ]:
# Add src to path if needed
import sys
from pathlib import Path
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import NB03 modules
from intake.strategy import (
    SessionStrategy,
    PromptBundle,
    QuestionWithIntent,
    build_session_strategy,
    build_prompt_bundle,
    build_session_strategy_and_bundle,
    determine_tone,
    get_tonal_guardrails,
    sort_questions_by_priority,
    get_mode_specific_goal,
    get_branch_conversational_approach,
)

from intake.safety import (
    SanitizedPacketView,
    sanitize_for_traveler,
    check_no_leakage,
    validate_traveler_safe_output,
    has_blocking_ambiguities,
    is_field_traveler_safe,
    is_field_internal_only,
    audit_packet_internal_data,
    sanitize_text_output,
)

# Also import NB02 for DecisionResult
from intake.decision import (
    DecisionResult,
    run_gap_and_decision,
)

# And NB01 for CanonicalPacket
from intake.packet_models import (
    CanonicalPacket,
    Slot,
    AuthorityLevel,
    EvidenceRef,
    Ambiguity,
    OwnerConstraint,
    SourceEnvelope,
)

print("NB03 modules imported successfully!")

## Demo: End-to-End Flow

Let's run through a complete example: raw text → packet → decision → strategy → prompts.

In [ ]:
# Step 1: Extract from raw text (NB01)
from intake.extractors import ExtractionPipeline

raw_text = """Family of 4 from Bangalore, wants to go to Singapore in March 2026.
Budget around 4L. Leisure trip."""

envelope = SourceEnvelope.from_freeform(raw_text, "agency_notes")
packet = ExtractionPipeline().extract([envelope])

print(f"Packet ID: {packet.packet_id}")
print(f"Facts extracted: {len(packet.facts)}")
dest = packet.facts.get('destination_candidates')
print(f"Destination: {dest.value if dest else 'N/A'}")

In [ ]:
# Step 2: Run gap and decision (NB02)
decision = run_gap_and_decision(packet)

print(f"Decision State: {decision.decision_state}")
print(f"Confidence: {decision.confidence_score:.2f}")
print(f"Hard Blockers: {decision.hard_blockers}")
print(f"Soft Blockers: {decision.soft_blockers}")

In [ ]:
# Step 3: Build session strategy (NB03)
strategy = build_session_strategy(decision, packet)

print(f"Session Goal: {strategy.session_goal}")
print(f"Suggested Tone: {strategy.suggested_tone}")
print(f"Suggested Opening: {strategy.suggested_opening}")
print(f"Priority Sequence: {strategy.priority_sequence}")

In [ ]:
# Step 4: Build prompt bundle (NB03)
bundle = build_prompt_bundle(strategy, decision, packet, audience="traveler")

print("=== USER MESSAGE ===")
print(bundle.user_message)
print("\n=== CONSTRAINTS ===")
for constraint in bundle.constraints:
    print(f"  - {constraint}")

## Demo: Traveler-Safe Sanitization

Show how internal-only data is structurally removed from traveler-facing output.

In [ ]:
# Create a packet with internal-only data
test_packet = CanonicalPacket(packet_id="test_safety")

test_packet.facts["destination_candidates"] = Slot(
    value=["Singapore"],
    confidence=0.9,
    authority_level=AuthorityLevel.EXPLICIT_USER,
    evidence_refs=[],
)

test_packet.facts["owner_constraints"] = Slot(
    value=[
        OwnerConstraint(
            text="Margin requirement: 15% markup",
            visibility="internal_only",
        ),
        OwnerConstraint(
            text="No supplier X due to payment issues",
            visibility="traveler_safe_transformable",
        ),
    ],
    confidence=1.0,
    authority_level=AuthorityLevel.EXPLICIT_OWNER,
    evidence_refs=[],
)

test_packet.hypotheses["trip_style"] = Slot(
    value="beach",
    confidence=0.5,
    authority_level=AuthorityLevel.SOFT_HYPOTHESIS,
    evidence_refs=[],
)

# Sanitize for traveler
sanitized = sanitize_for_traveler(test_packet)

print("Original packet facts:", list(test_packet.facts.keys()))
print("Sanitized facts:", list(sanitized.facts.keys()))
print("Original hypotheses:", list(test_packet.hypotheses.keys()))
print("Sanitized hypotheses: (removed structurally)")

In [ ]:
# Check for leakage
bad_bundle = {
    "user_message": "Based on my hypothesis that you like beaches, I suggest Andaman.",
    "system_context": "Decision state is ASK_FOLLOWUP with high confidence.",
}

leaks = check_no_leakage(bad_bundle)
print(f"Leakage detected: {len(leaks)} issues")
for leak in leaks:
    print(f"  - {leak}")

## Demo: All 5 Decision States

In [ ]:
def demo_decision_state(state_name: str):
    pkt = CanonicalPacket(packet_id=f"demo_{state_name}")
    decision = DecisionResult(
        packet_id=pkt.packet_id,
        current_stage="discovery",
        operating_mode="normal_intake",
        decision_state=state_name,
        confidence_score=0.7,
    )
    strategy = build_session_strategy(decision, pkt)
    print(f"\n=== {state_name} ===")
    print(f"Goal: {strategy.session_goal}")
    print(f"Opening: {strategy.suggested_opening}")

for state in ["ASK_FOLLOWUP", "PROCEED_INTERNAL_DRAFT", "PROCEED_TRAVELER_SAFE", "BRANCH_OPTIONS", "STOP_NEEDS_REVIEW"]:
    demo_decision_state(state)

## Demo: Tone Scaling

In [ ]:
for confidence in [0.1, 0.3, 0.6, 0.9]:
    tone = determine_tone(confidence)
    guardrails = get_tonal_guardrails(tone)
    print(f"\nConfidence {confidence:.1f} → Tone: {tone}")
    print(f"  Guardrails: {guardrails[:2]}")

## Demo: Operating Mode Behaviors

In [ ]:
modes = ["normal_intake", "audit", "emergency", "follow_up", "cancellation", "post_trip", "coordinator_group", "owner_review"]

print("ASK_FOLLOWUP goals by operating mode:")
for mode in modes:
    goal = get_mode_specific_goal("ASK_FOLLOWUP", mode)
    print(f"  {mode:20s}: {goal[:60]}...")

## Convenience Entry Point

Primary NB03 entry point: `build_session_strategy_and_bundle()`

In [ ]:
strategy, bundle = build_session_strategy_and_bundle(
    decision=decision,
    packet=packet,
    audience="traveler",
)

print(f"Strategy type: {type(strategy).__name__}")
print(f"Bundle type: {type(bundle).__name__}")